# Fetching and parsing in Python

---
**Author**: Marko Bajec

**Last update**: 3.3.2026

**Description**: in this notebook I describe a selection of libraries that might come handy for **fetching** and **parsing** pages as well as for **text analysis** including **natural language processing**. Many of these libraries are required in other notebooks that I share for this course. The libraries discussed are:
* <code>urllib</code>: [urllib](https://docs.python.org/3/library/urllib.html#module-urllib) is a package of modules for working with URLs,
* <code>beautifulsoup4</code>: [beautifulsoap](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) is a library for pulling data out of HTML and XML files. Works with DOM,
* <code>nltk</code>: [nltk](https://www.nltk.org) is a natural language processing toolkit, 
* <code>polyglot</code>: [polyglot](https://polyglot.readthedocs.io/en/latest/) is another library for natural language processing which includes support for Slovenian language,
* <code>re</code>: [re]() is a library for working with regular expressions

As from 2020, [Stanza](https://pypi.org/project/stanza/) took the lead as the **NLP toolkit** which also supports Slovenian language - check its fork [Classla](https://github.com/clarinsi/classla). More information can be found [here](https://www.clarin.si/info/k-centre/). 

---

Let's import all modules at once.

In [ ]:
%pip install nltk beautifulsoup4

In [1]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer
from nltk.stem import PorterStemmer
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import WordNetLemmatizer
#import polyglot
import re
import string
import urllib
import urllib.request
from bs4 import BeautifulSoup

## 1. Fetching pages
### 1.1 Simple http request

In [2]:
# send http request 
f = urllib.request.urlopen('http://www.delo.si')
# print status code, URL, page info, and payload limitted to 1000 bytes
print('Status code:', f.getcode())
print('URL:', f.geturl())
#print(f.info())
print(f.read(1000).decode('utf-8'))

Status code: 200
URL: https://www.delo.si/
<!DOCTYPE html><html lang="sl"><head><meta charSet="utf-8"/><link rel="canonical" href="https://www.delo.si"/><link rel="preconnect" href="https://sdk.privacy-center.org" crossorigin="anonymous"/><link rel="preconnect" href="//script.dotmetrics.net" crossorigin="anonymous"/><link rel="preconnect" href="https://cdnjs.cloudflare.com" crossorigin="anonymous"/><link rel="preconnect" href="https://stgm.delo.si" crossorigin="anonymous"/><link rel="preconnect" href="https://www.google.com" crossorigin="anonymous"/><link rel="preconnect" href="https://services.delo.si" crossorigin="anonymous"/><link rel="preconnect" href="https://connect.facebook.net" crossorigin="anonymous"/><link rel="preconnect" href="https://ajax.googleapis.com" crossorigin="anonymous"/><link rel="preconnect" href="https://fonts.googleapis.com" crossorigin="anonymous"/><link rel="preconnect" href="https://fonts.google.com" crossorigin="anonymous"/><link rel="preconnect" href="http

What we can see is that simple fetching does not suffice for pulling out all the data if dynamic pages are used. To render a typical page, a browser would usually execute several additional HTTP requests to download all the required files. Moreover, modern web pages are dynamic, i.e. they use scripting languages to pull data from a database and then render it on the page. 

### Headless browsers
To get a page that corresponds to one that we see in a browser when following a certain URL, one could use a **headless browser**. In short, headless browsers are web browsers without a graphical user interface and are usually controlled programmatically or via a command-line interface. They are used in several contexts but most notably for **automatic usability testing** and **web scraping**. In web extration we need a DOM representaion of the exact content as it is rendered in a full browser.

To run a headless browser in Python one option is to use **Headless Chrome** together with **Selenium**. See [here](https://duo.com/decipher/driving-headless-chrome-with-python) for instructions of how to install and use Headless Chrome with Python.  

### 1.2 http HEAD request using request object
You will notice that changing <code>GET</code> to <code>HEAD</code> will have no effect. The reason for this is that the command <code>page = resp.read(1000)...</code>, which is essentially reading the first 1000 bytes of the response body. The HEAD method is supposed to retrieve only the headers, not the actual content. To do that and avoid retrieving the content, you should refrain from reading the response body.

In [3]:
req = urllib.request.Request("http://www.times.si", method="HEAD")
resp = urllib.request.urlopen(req)

headers=resp.getheaders()
print(headers)

page = resp.read(1000).decode('utf-8')
print(resp.geturl())
print(resp.info())
print(resp.getcode())
print(len(page))
print(page)

[('Server', 'nginx/1.14.0 (Ubuntu)'), ('Date', 'Tue, 03 Mar 2026 09:22:04 GMT'), ('Content-Type', 'text/html; charset=utf-8'), ('Content-Length', '45511'), ('Last-Modified', 'Tue, 03 Mar 2026 09:22:03 GMT'), ('Connection', 'close'), ('ETag', '"69a6a83b-b1c7"'), ('Expires', 'Tue, 03 Mar 2026 09:23:03 GMT'), ('Cache-Control', 'max-age=59'), ('Accept-Ranges', 'bytes')]
https://www.times.si/
Server: nginx/1.14.0 (Ubuntu)
Date: Tue, 03 Mar 2026 09:22:04 GMT
Content-Type: text/html; charset=utf-8
Content-Length: 45511
Last-Modified: Tue, 03 Mar 2026 09:22:03 GMT
Connection: close
ETag: "69a6a83b-b1c7"
Expires: Tue, 03 Mar 2026 09:23:03 GMT
Cache-Control: max-age=59
Accept-Ranges: bytes


200
994
<!doctype html><html lang="sl"><head><!-- Google tag (gtag.js) --><script async src="https://www.googletagmanager.com/gtag/js?id=G-VLKSG5FWDE"></script><script>window.dataLayer = window.dataLayer || [];function gtag(){dataLayer.push(arguments);}gtag('js', new Date()); gtag('config', 'G-VLKSG5FWDE');<

You can get a web page header also with <code>curl</code>. Use switch <code>-I</code>

In [4]:
!curl -I http://www.times.si

HTTP/1.1 301 Moved Permanently
Server: nginx/1.14.0 (Ubuntu)
Date: Tue, 03 Mar 2026 09:22:30 GMT
Content-Type: text/html
Content-Length: 194
Connection: keep-alive
Location: https://www.times.si/



  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
  0    194   0      0   0      0      0      0                              0
  0    194   0      0   0      0      0      0                              0
  0    194   0      0   0      0      0      0                              0


### Extracting text
This simple code shows how to extract text parts from a web page. It uses <code>beautifulsoap</code> library which cleans up <code>HTTP request payload</code> and creates a DOM tree.

In [5]:
# This is an example of code which extracts all text from a web page 
url_address = "http://www.times.si"

with urllib.request.urlopen(url_address) as url:
    html = url.read()

# Uncomment to see what you get by default
print("Normal HTML")
print("-------------------------------------------------------------")
print(html[:1000])
print("\n\n")

print("Beautified HTML")
print("-------------------------------------------------------------")
soup = BeautifulSoup(html)
#print(soup)

# kill all script and style elements
for script in soup(["script", "style"]):
    script.extract()    # rip it out

# get text
text = soup.get_text()

# break into lines and remove leading and trailing space on each
lines = (line.strip() for line in text.splitlines())
# break multi-headlines into a line each
chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
# drop blank lines
text = '\n'.join(chunk for chunk in chunks if chunk)

print(text)

print("FIRST 10 LINKS")
print("-------------------------------------------------------------")
links = soup.find_all('a')
for l in links[1:10]:
    print(l)

Normal HTML
-------------------------------------------------------------
b'<!doctype html><html lang="sl"><head><!-- Google tag (gtag.js) --><script async src="https://www.googletagmanager.com/gtag/js?id=G-VLKSG5FWDE"></script><script>window.dataLayer = window.dataLayer || [];function gtag(){dataLayer.push(arguments);}gtag(\'js\', new Date()); gtag(\'config\', \'G-VLKSG5FWDE\');</script><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1, shrink-to-fit=no"><link rel="stylesheet" href="https://stackpath.bootstrapcdn.com/bootstrap/4.3.1/css/bootstrap.min.css" integrity="sha384-ggOyR0iXCbMQv3Xipma34MD+dH/1fQ784/j6cY/iJTQUOhcWr7x9JvoRxT2MZw1T" crossorigin="anonymous"><link rel=stylesheet href="/s/css/times.css"><title>Vse novice na enem mestu - TIMES.si</title><meta name="keywords" content="novice,sve\xc5\xbee novice,zadnje novice,slovenija,\xc5\xa1port,gospodarstvo,svet,evropa,smrt,nesre\xc4\x8da,tehnologija,24ur,rtvslo" /><meta name="description" con

## 2. Processing text with NLTK
NLTK stands for **Natural Language Processing Toolkit**. It is a Python library that supports numerous tasks that are common in NLP, such as *tokenization*, *lemmatization*, *stemming*, *pos tagging*, *semantic reasoning*, *parsing*, etc. It provides interfaces to many corpora and lexical resources such as *WordNet*.

Below are few examples of how to use NLTK. For complete documentation see [NLTK webpage](https://www.nltk.org).

### 2.1. Tokenization
Suggested reading: [The art of tokenization](https://www.ibm.com/developerworks/community/blogs/nlp/entry/tokenization?lang=en)

In [6]:
# simple tokenization
nltk.download('punkt')
nltk.download('punkt_tab')
sentence = "You can\'t say you didn\'t know this was wrong! And yet you did it. I want you to read these books. "
sentence += "This book was written in 1998 by Dan Taylor. It is about information extraction from web sources."
sentence += " If you haven't read it yet then do so ASAP."
sentence = sentence.lower()
tokens = nltk.word_tokenize(sentence)
print(tokens)

['you', 'ca', "n't", 'say', 'you', 'did', "n't", 'know', 'this', 'was', 'wrong', '!', 'and', 'yet', 'you', 'did', 'it', '.', 'i', 'want', 'you', 'to', 'read', 'these', 'books', '.', 'this', 'book', 'was', 'written', 'in', '1998', 'by', 'dan', 'taylor', '.', 'it', 'is', 'about', 'information', 'extraction', 'from', 'web', 'sources', '.', 'if', 'you', 'have', "n't", 'read', 'it', 'yet', 'then', 'do', 'so', 'asap', '.']


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\marko\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\marko\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


### 2.2. Contractions

In [7]:
# handling contractions, e.g. don't --> do not

# example of contractions dictionary - incomplete!
contractions_dict = { 
"can't": "cannot",
"didn't": "did not",
"don't": "do not",
"isn't": "is not",
"haven't": "have not"
}
contractions_re = re.compile('(%s)' % '|'.join(contractions_dict.keys()))

def expand_contractions(s, contractions_dict=contractions_dict):
     def replace(match):
         return contractions_dict[match.group(0)]
     return contractions_re.sub(replace, s)

tokens = nltk.word_tokenize(expand_contractions(sentence))

print(tokens)

['you', 'can', 'not', 'say', 'you', 'did', 'not', 'know', 'this', 'was', 'wrong', '!', 'and', 'yet', 'you', 'did', 'it', '.', 'i', 'want', 'you', 'to', 'read', 'these', 'books', '.', 'this', 'book', 'was', 'written', 'in', '1998', 'by', 'dan', 'taylor', '.', 'it', 'is', 'about', 'information', 'extraction', 'from', 'web', 'sources', '.', 'if', 'you', 'have', 'not', 'read', 'it', 'yet', 'then', 'do', 'so', 'asap', '.']


### 2.3. Stopwords

In [8]:
# print stopwords for English
nltk.download('stopwords')
print(stopwords.words('slovene'))

['ali', 'ampak', 'bodisi', 'in', 'kajti', 'marveč', 'namreč', 'ne', 'niti', 'oziroma', 'pa', 'saj', 'sicer', 'temveč', 'ter', 'toda', 'torej', 'vendar', 'vendarle', 'zakaj', 'če', 'čeprav', 'čeravno', 'četudi', 'čim', 'da', 'kadar', 'kakor', 'ker', 'ki', 'ko', 'kot', 'naj', 'najsi', 'odkar', 'preden', 'dve', 'dvema', 'dveh', 'šest', 'šestdeset', 'šestindvajset', 'šestintrideset', 'šestnajst', 'šeststo', 'štiri', 'štirideset', 'štiriindvajset', 'štirinajst', 'štiristo', 'deset', 'devet', 'devetdeset', 'devetintrideset', 'devetnajst', 'devetsto', 'dvainšestdeset', 'dvaindvajset', 'dvajset', 'dvanajst', 'dvesto', 'enaindvajset', 'enaintrideset', 'enajst', 'nič', 'osem', 'osemdeset', 'oseminštirideset', 'osemindevetdeset', 'osemnajst', 'pet', 'petdeset', 'petinštirideset', 'petindevetdeset', 'petindvajset', 'petinosemdeset', 'petinpetdeset', 'petinsedemdeset', 'petintrideset', 'petnajst', 'petsto', 'sedem', 'sedemdeset', 'sedeminšestdeset', 'sedemindvajset', 'sedeminpetdeset', 'sedemnajst'

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\marko\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [9]:
# remove stopwords
filtered_words = [word for word in tokens if word not in stopwords.words('english')]

print(filtered_words)

['say', 'know', 'wrong', '!', 'yet', '.', 'want', 'read', 'books', '.', 'book', 'written', '1998', 'dan', 'taylor', '.', 'information', 'extraction', 'web', 'sources', '.', 'read', 'yet', 'asap', '.']


### 2.4. Punctation

In [10]:
nonPunct = re.compile('.*[A-Za-z0-9].*')  # must contain a letter or digit
words_no_punct = [w for w in filtered_words if nonPunct.match(w)]

print(words_no_punct)

['say', 'know', 'wrong', 'yet', 'want', 'read', 'books', 'book', 'written', '1998', 'dan', 'taylor', 'information', 'extraction', 'web', 'sources', 'read', 'yet', 'asap']


### 2.5. Using explicit RE pattern in tokenizer

In [ ]:
# This example shows how to use RegexpTokenizer
# Function preprocess does all: makes the sentence lowercase, creates tokens, removes stopwords and punctation
# and then puts tokens back to a sentence

def preprocess(sentence):
    sentence = sentence.lower()
    tokenizer = RegexpTokenizer(r'\w+')
    tokens = tokenizer.tokenize(sentence)
    filtered_words = [w for w in tokens if not w in stopwords.words('english')]
    return " ".join(filtered_words)

sentence = "You can\'t say you didn\'t know! I told you several times and you know that."
print(preprocess(sentence))

### 2.6. POS tagging
Part-of-speech tagging (POS tagging) is grammatical analysis which marks each word in a text (corpus) as corresponding to a particular part of speech, e.g. <code>noun</code>, <code>verb</code>, <code>adjective</code>, <code>adverb</code>, etc. 

For details on *Categorizing and Tagging Wordshttp* check [here](www.nltk.org/book/ch05.html).

In [11]:
nltk.download('averaged_perceptron_tagger_eng')

wordsPOS = nltk.pos_tag(words_no_punct)
# I am creating this set for for the use in the lemmatizer
wordsPOSset = {}
for i in range (0, len(wordsPOS)):
    wordsPOSset.update({(wordsPOS[i][0]):wordsPOS[i][1][0]})
    print(wordsPOS[i][0] + ":" + wordsPOS[i][1])

say:VB
know:VBP
wrong:JJ
yet:RB
want:VBP
read:JJ
books:NNS
book:NN
written:VBN
1998:CD
dan:JJ
taylor:NN
information:NN
extraction:NN
web:NN
sources:NNS
read:VBP
yet:RB
asap:JJ


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\marko\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


### 2.7. Stemming

In [12]:
# this is a simple stemming
ps = PorterStemmer()
 
for word in words_no_punct:
    print(word + ":" + ps.stem(word))

say:say
know:know
wrong:wrong
yet:yet
want:want
read:read
books:book
book:book
written:written
1998:1998
dan:dan
taylor:taylor
information:inform
extraction:extract
web:web
sources:sourc
read:read
yet:yet
asap:asap


### 2.8. Lemmatization

In [13]:
# This example shows how to use NLTK lemmatizer. Remember that you have to tell what is the word you 
# would like to lemmatize. By default the lemmatizer expects a noun. But it could verb, adverb, adjecive and so on. 

# Pos tagger was trained on treebank corpora. This function maps treebank tags 
# into wordnet tags as expected in the lemmatizer.
nltk.download('wordnet')
from nltk.corpus import wordnet
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        #if not J, V, N or R then make it default, i.e. "n" as naun 
        return 'n'

lemmatizer = WordNetLemmatizer()
for word in words_no_punct:
    print(word + ":" + lemmatizer.lemmatize(word, pos=get_wordnet_pos(wordsPOSset[word])))

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\marko\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


say:say
know:know
wrong:wrong
yet:yet
want:want
read:read
books:book
book:book
written:write
1998:1998
dan:dan
taylor:taylor
information:information
extraction:extraction
web:web
sources:source
read:read
yet:yet
asap:asap


### 2.9. Language detection

There are many Python modules available that offer a model for language detection. Most known modules probably are:
<code>fasttext</code>, <code>polyglot</code>, <code>langdetect</code>, <code>langid</code> etc. They differentiate in the ease of use, accuracy and efficiency.

**Important**: !pip my point to wrong pip - use these instead:  

In [ ]:
import sys
!{sys.executable} -m pip install -U langdetect langid

In [20]:
import langdetect
from langdetect import detect

text = "To je preprost jezik, ki ga govori le malo ljudi."
language = detect(text)
print(language)

hr


In [21]:
import langid

text_fr = "Ceci est un texte en français."
text_sl = "To je preprost jezik, ki ga govori le malo ljudi"
language, confidence = langid.classify(text_fr)
print(language)  # Output: fr
print(confidence)  # Output: Confidence score
language, confidence = langid.classify(text_sl)
print(language)  # Output: fr
print(confidence)  # Output: Confidence score

fr
-158.1863522529602
sl
-77.15520524978638
